In [2]:

import zipfile

def unzip_file(zip_path, extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

unzip_file('/content/abalone.zip', '/content/')

In [3]:
def load_data(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            row = line.strip().split(',')
            data.append(row)
    return data

data = load_data('/content/abalone.data')

In [4]:
def encode_sex(value):
    if value == 'M':
        return 0.0
    elif value == 'F':
        return 1.0
    else:
        return 0.5

def preprocess(data):
    X = []
    Y = []

    for row in data:
        features = [encode_sex(row[0])]
        for i in range(1, 8):
            features.append(float(row[i]))
        X.append(features)
        Y.append(float(row[8]))  # Rings

    return X, Y

X, Y = preprocess(data)

In [5]:
def min_max_normalize(X):
    mins = []
    maxs = []

    for j in range(len(X[0])):
        min_val = X[0][j]
        max_val = X[0][j]
        for i in range(len(X)):
            if X[i][j] < min_val:
                min_val = X[i][j]
            if X[i][j] > max_val:
                max_val = X[i][j]
        mins.append(min_val)
        maxs.append(max_val)

    X_norm = []
    for i in range(len(X)):
        row = []
        for j in range(len(X[0])):
            if maxs[j] - mins[j] == 0:
                row.append(0)
            else:
                row.append((X[i][j] - mins[j]) / (maxs[j] - mins[j]))
        X_norm.append(row)

    return X_norm

X = min_max_normalize(X)



In [6]:
def train_test_split(X, Y, ratio=0.8):
    split = int(len(X) * ratio)
    return X[:split], X[split:], Y[:split], Y[split:]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y)

In [7]:
import numpy as np
def sigmoid(x):
    return 1 / (1 + (2.71828 ** (-x)))

def sigmoid_derivative(output):
    return output * (1 - output)
def relu(x):
    if x > 0:
        return x
    else:
        return 0


In [8]:

def relu_derivative(x):
    if x > 0:
        return 1
    else:
        return 0

In [9]:
import random

def initialize_weights(input_size, hidden_size):
    w_ih = []
    for i in range(hidden_size):
        row = []
        for j in range(input_size):
            row.append(random.uniform(-0.5, 0.5))
        w_ih.append(row)

    w_ho = []
    for i in range(hidden_size):
        w_ho.append(random.uniform(-0.5, 0.5))

    return w_ih, w_ho


In [10]:
def forward_pass(inputs, w_ih, w_ho):
    hidden = []

    for i in range(len(w_ih)):
        total = 0
        for j in range(len(inputs)):
            total += inputs[j] * w_ih[i][j]
        hidden.append(sigmoid(total))

    output = 0
    for i in range(len(hidden)):
        output += hidden[i] * w_ho[i]

    return hidden, output



In [11]:
def backpropagation(inputs, hidden, output, target, w_ih, w_ho, lr):
    error = target - output
    delta_output = error  # linear output

    # hidden → output update
    for i in range(len(w_ho)):
        w_ho[i] += lr * delta_output * hidden[i]

    # input → hidden update
    for i in range(len(w_ih)):
        for j in range(len(w_ih[i])):
            delta_hidden = sigmoid_derivative(hidden[i]) * w_ho[i] * delta_output
            w_ih[i][j] += lr * delta_hidden * inputs[j]


In [12]:
def train(X, Y, epochs, lr):
    input_size = len(X[0])
    hidden_size = 12  # ideal for sigmoid

    w_ih, w_ho = initialize_weights(input_size, hidden_size)

    for epoch in range(epochs):
        mse = 0
        for i in range(len(X)):
            hidden, output = forward_pass(X[i], w_ih, w_ho)
            mse += (Y[i] - output) ** 2
            backpropagation(X[i], hidden, output, Y[i], w_ih, w_ho, lr)

        if epoch % 20 == 0:
            print("Epoch:", epoch, "MSE:", mse / len(X))

    return w_ih, w_ho

In [13]:
def predict(X, w_ih, w_ho):
    preds = []
    for x in X:
        _, output = forward_pass(x, w_ih, w_ho)
        preds.append(output)
    return preds

In [14]:
weights_ih, weights_ho = train(
    X_train,
    Y_train,
    epochs=500,
    lr=0.003
)

preds = predict(X_test[:5], weights_ih, weights_ho)

print("Predicted Rings:", preds)
print("Actual Rings:", Y_test[:5])

Epoch: 0 MSE: 7.772946842414447
Epoch: 20 MSE: 4.092227558221384
Epoch: 40 MSE: 3.6411895637940392
Epoch: 60 MSE: 3.554538000739679
Epoch: 80 MSE: 3.5080431636045604
Epoch: 100 MSE: 3.480412307089823
Epoch: 120 MSE: 3.4665747803625724
Epoch: 140 MSE: 3.462037289649424
Epoch: 160 MSE: 3.4617929048611304
Epoch: 180 MSE: 3.4629684715380877
Epoch: 200 MSE: 3.464424007059227
Epoch: 220 MSE: 3.465730176092605
Epoch: 240 MSE: 3.466719342058741
Epoch: 260 MSE: 3.4673476468719295
Epoch: 280 MSE: 3.4676360242042
Epoch: 300 MSE: 3.4676347994274708
Epoch: 320 MSE: 3.4674063475658783
Epoch: 340 MSE: 3.4670177653864322
Epoch: 360 MSE: 3.4665343781335958
Epoch: 380 MSE: 3.4660114641303053
Epoch: 400 MSE: 3.46548794004183
Epoch: 420 MSE: 3.4649857336928966
Epoch: 440 MSE: 3.464514080507073
Epoch: 460 MSE: 3.4640751183760496
Epoch: 480 MSE: 3.463668042296498
Predicted Rings: [12.650308523781106, 10.173572629825816, 11.368432987280581, 12.951328408924496, 13.203044244463152]
Actual Rings: [12.0, 14.0, 1

In [15]:
def rings_to_age(rings):
    return rings + 1.5

print("Predicted Ages:", [rings_to_age(p) for p in preds])

Predicted Ages: [14.150308523781106, 11.673572629825816, 12.868432987280581, 14.451328408924496, 14.703044244463152]


In [16]:
def mean_absolute_error(actual, predicted):
    total = 0
    for i in range(len(actual)):
        total += abs(actual[i] - predicted[i])
    return total / len(actual)

In [17]:
def root_mean_squared_error(actual, predicted):
    total = 0
    for i in range(len(actual)):
        total += (actual[i] - predicted[i]) ** 2
    return (total / len(actual)) ** 0.5



In [18]:
def regression_accuracy(actual, predicted):
    mae = mean_absolute_error(actual, predicted)

    mean_actual = 0
    for v in actual:
        mean_actual += v
    mean_actual = mean_actual / len(actual)

    accuracy = 100 - (mae / mean_actual) * 100
    return accuracy

In [19]:
test_preds = predict(X_test, weights_ih, weights_ho)

mae = mean_absolute_error(Y_test, test_preds)
rmse = root_mean_squared_error(Y_test, test_preds)
acc = regression_accuracy(Y_test, test_preds)

print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("Regression Accuracy (%):", acc)

Mean Absolute Error (MAE): 3.1985455090235546
Root Mean Squared Error (RMSE): 3.503879494724252
Regression Accuracy (%): 66.36074920689782


In [20]:
def print_comparison_table(actual, predicted, rows=10):
    print("Index\tActual Rings\tPredicted Rings")
    print("---------------------------------------")

    for i in range(rows):
        print(i, "\t", round(actual[i], 2), "\t\t", round(predicted[i], 2))

print_comparison_table(Y_test, test_preds, rows=10)


Index	Actual Rings	Predicted Rings
---------------------------------------
0 	 12.0 		 12.65
1 	 14.0 		 10.17
2 	 13.0 		 11.37
3 	 13.0 		 12.95
4 	 12.0 		 13.2
5 	 14.0 		 13.12
6 	 11.0 		 10.05
7 	 13.0 		 11.46
8 	 10.0 		 11.89
9 	 11.0 		 12.52


In [21]:
def print_age_table(actual_rings, predicted_rings, rows=10):
    print("Index\tActual Age\tPredicted Age")
    print("-----------------------------------")

    for i in range(rows):
        actual_age = actual_rings[i] + 1.5
        predicted_age = predicted_rings[i] + 1.5
        print(i, "\t", round(actual_age, 2), "\t\t", round(predicted_age, 2))

print_age_table(Y_test, test_preds, rows=10)


Index	Actual Age	Predicted Age
-----------------------------------
0 	 13.5 		 14.15
1 	 15.5 		 11.67
2 	 14.5 		 12.87
3 	 14.5 		 14.45
4 	 13.5 		 14.7
5 	 15.5 		 14.62
6 	 12.5 		 11.55
7 	 14.5 		 12.96
8 	 11.5 		 13.39
9 	 12.5 		 14.02
